# 🧱 Lab: Tool-Using Agent

**Module 7: AI Agents** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Build** a complete tool-using agent with multiple real tools, a ReAct-style reasoning loop, and conversation memory
2. **Implement** planning and task decomposition so the agent breaks complex requests into manageable steps
3. **Apply** reflection so the agent evaluates its own answer before presenting it to the user
4. **Integrate** error recovery, stop conditions, and guardrails to make the agent robust and safe

## Concepts Covered

| Concept | From |
|---------|------|
| Function Calling | mini-function-call |
| Tool Execution | mini-calculator |
| Agent Loop | mini-agent-loop |
| ReAct Framework | mini-react |
| Agent Memory (Short-Term) | mini-memory |
| Planning & Task Decomposition | This lab |
| Reflection | This lab |
| Stop Conditions | This lab |
| Error Recovery | This lab |
| Agent Guardrails | This lab |

## Prerequisites

- **mini-function-call**: Defining tools with JSON schemas and completing a function-calling round trip
- **mini-calculator**: Extracting parameters, executing tools, and parsing results back to the LLM
- **mini-agent-loop**: Building a loop that repeatedly calls the LLM and dispatches tools until done
- **mini-react**: ReAct prompting for transparent reasoning traces, and Plan-and-Execute as an alternative
- **mini-memory**: Short-term (session) and long-term (persistent) conversation memory

## Setup

In [1]:
import os
import json
import math
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv()

client = OpenAI()

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## What We're Building

In the mini-labs, we built each piece separately: function calling, a basic loop, ReAct reasoning, and memory. Now we combine everything into a **complete research assistant agent** that can:

1. **Plan** — break a complex question into sub-tasks before acting
2. **Reason & Act** — use ReAct-style thinking at each step
3. **Use tools** — call multiple tools (calculator, unit converter, knowledge lookup)
4. **Recover from errors** — handle tool failures gracefully instead of crashing
5. **Reflect** — evaluate its own answer before presenting it
6. **Respect guardrails** — refuse disallowed requests
7. **Stop safely** — enforce a maximum iteration limit

We'll build this incrementally, testing each capability as we add it.

## Part 1: Define the Tool Suite

Our agent needs tools that are useful *and* that can fail (so we can practice error recovery). We define three tools:

- **calculate** — math operations (can fail on division by zero or unknown operations)
- **convert_units** — unit conversions (can fail on unsupported pairs)
- **lookup_fact** — a small knowledge base (can fail when a topic isn't found)

In [2]:
# --- Tool implementations ---

def calculate(expression):
    """Evaluate a math expression safely. Supports +, -, *, /, **, sqrt, round."""
    allowed = set("0123456789+-*/.() ")
    # Allow named math functions
    clean = expression
    for fn_name in ["sqrt", "round", "abs", "pow"]:
        clean = clean.replace(fn_name, "")
    if not all(ch in allowed for ch in clean):
        return json.dumps({"error": f"Unsafe expression: {expression}"})
    try:
        safe_dict = {"sqrt": math.sqrt, "round": round, "abs": abs, "pow": pow}
        result = eval(expression, {"__builtins__": {}}, safe_dict)
        return json.dumps({"expression": expression, "result": round(result, 6)})
    except ZeroDivisionError:
        return json.dumps({"error": "Division by zero"})
    except Exception as e:
        return json.dumps({"error": str(e)})


def convert_units(value, from_unit, to_unit):
    """Convert between common units."""
    conversions = {
        ("miles", "km"): lambda v: v * 1.60934,
        ("km", "miles"): lambda v: v / 1.60934,
        ("fahrenheit", "celsius"): lambda v: (v - 32) * 5 / 9,
        ("celsius", "fahrenheit"): lambda v: v * 9 / 5 + 32,
        ("pounds", "kg"): lambda v: v * 0.453592,
        ("kg", "pounds"): lambda v: v / 0.453592,
        ("meters", "feet"): lambda v: v * 3.28084,
        ("feet", "meters"): lambda v: v / 3.28084,
    }
    key = (from_unit.lower(), to_unit.lower())
    fn = conversions.get(key)
    if fn is None:
        return json.dumps({"error": f"Unsupported conversion: {from_unit} → {to_unit}"})
    result = round(fn(value), 4)
    return json.dumps({"value": value, "from": from_unit, "to": to_unit, "result": result})


def lookup_fact(topic):
    """Look up a fact from a small knowledge base."""
    kb = {
        "speed of light": "The speed of light in vacuum is 299,792,458 meters per second.",
        "earth radius": "Earth's mean radius is approximately 6,371 kilometers.",
        "water boiling point": "Water boils at 100°C (212°F) at standard atmospheric pressure.",
        "moon distance": "The average distance from Earth to the Moon is about 384,400 kilometers.",
        "pi": "Pi (π) is approximately 3.14159265358979.",
        "marathon distance": "A marathon is 42.195 kilometers (26.2 miles).",
    }
    result = kb.get(topic.lower())
    if result is None:
        return json.dumps({"error": f"Topic '{topic}' not found. Available: {list(kb.keys())}"})
    return json.dumps({"topic": topic, "fact": result})


# Map names to functions
TOOL_FUNCTIONS = {
    "calculate": calculate,
    "convert_units": convert_units,
    "lookup_fact": lookup_fact,
}

print("Tools defined: calculate, convert_units, lookup_fact")

Tools defined: calculate, convert_units, lookup_fact


In [3]:
# --- Tool schemas for OpenAI ---

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a math expression. Supports +, -, *, /, **, sqrt(), round(), abs(), pow(). Example: 'sqrt(144) + 3 * 2'",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The math expression to evaluate."
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "convert_units",
            "description": "Convert a value between units. Supported pairs: miles/km, fahrenheit/celsius, pounds/kg, meters/feet.",
            "parameters": {
                "type": "object",
                "properties": {
                    "value": {"type": "number", "description": "The numeric value to convert."},
                    "from_unit": {"type": "string", "description": "The source unit."},
                    "to_unit": {"type": "string", "description": "The target unit."}
                },
                "required": ["value", "from_unit", "to_unit"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_fact",
            "description": "Look up a fact from the knowledge base. Available topics: speed of light, earth radius, water boiling point, moon distance, pi, marathon distance.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string", "description": "The topic to look up."}
                },
                "required": ["topic"]
            }
        }
    }
]

print(f"{len(TOOLS)} tool schemas defined.")

3 tool schemas defined.


Let's quickly verify the tools work — including their error paths.

In [4]:
# Successful calls
print(calculate("sqrt(144) + 10"))
print(convert_units(100, "miles", "km"))
print(lookup_fact("moon distance"))

# Error cases
print(calculate("10 / 0"))                          # division by zero
print(convert_units(5, "liters", "gallons"))        # unsupported
print(lookup_fact("population of mars"))             # not found

{"expression": "sqrt(144) + 10", "result": 22.0}
{"value": 100, "from": "miles", "to": "km", "result": 160.934}
{"topic": "moon distance", "fact": "The average distance from Earth to the Moon is about 384,400 kilometers."}
{"error": "Division by zero"}
{"error": "Unsupported conversion: liters \u2192 gallons"}
{"error": "Topic 'population of mars' not found. Available: ['speed of light', 'earth radius', 'water boiling point', 'moon distance', 'pi', 'marathon distance']"}


All three tools return JSON with an `"error"` key when something goes wrong — the agent will need to handle these gracefully.

## Part 2: The System Prompt — ReAct + Planning + Reflection + Guardrails

The system prompt is where we combine everything. Recall from `mini-react` that the prompting strategy is what turns a basic loop into a ReAct agent. Here we extend it with **planning**, **reflection**, and **guardrails**.

In [5]:
SYSTEM_PROMPT = """You are a research assistant that solves problems step by step.

## WORKFLOW

For every user request, follow this workflow:

1. **PLAN**: Break the problem into numbered sub-tasks.
2. **EXECUTE** (ReAct style): For each sub-task:
   - Thought: Explain what you need to do and why.
   - Action: Call the appropriate tool.
   - Observation: Note the result.
   If a tool returns an error, explain the issue and try an alternative approach.
3. **REFLECT**: After all sub-tasks are done, review your work:
   - Did you answer the original question fully?
   - Are the numbers correct and units consistent?
   - If something looks wrong, redo that step.
4. **ANSWER**: Provide the final answer clearly.

Always show your plan, thoughts, and reflection in your message content.

## GUARDRAILS

You MUST refuse the following types of requests:
- Executing system commands or accessing files
- Generating harmful, illegal, or unethical content
- Anything unrelated to math, conversions, or factual lookups

If a request violates these guardrails, respond with a polite refusal explaining why.

## ERROR HANDLING

If a tool returns an error:
- Do NOT give up immediately
- Explain what went wrong
- Try a different approach if possible (e.g., rephrase, use a different tool)
- If no workaround exists, report the failure clearly to the user
"""

print("System prompt defined.")

System prompt defined.


## Part 3: The Agent Loop with Error Recovery & Stop Conditions

This is the core of the lab — a robust agent loop that combines everything. Compared to `mini-agent-loop`, we add:

- **Max iterations** (stop condition) to prevent infinite loops
- **Error recovery** — tool errors are returned to the LLM so it can adapt
- **Reasoning trace** captured for inspection
- **Short-term memory** (the messages list persists across calls)

In [6]:
class ResearchAgent:
    """A tool-using agent with ReAct reasoning, planning, reflection,
    error recovery, stop conditions, and guardrails."""
    
    def __init__(self, max_iterations=15):
        self.messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        self.max_iterations = max_iterations
        self.trace = []  # reasoning trace for the latest query
    
    def _execute_tool_call(self, tool_call):
        """Execute a single tool call with error recovery."""
        name = tool_call.function.name
        
        # Parse arguments
        try:
            args = json.loads(tool_call.function.arguments)
        except json.JSONDecodeError as e:
            return json.dumps({"error": f"Invalid arguments JSON: {e}"})
        
        # Look up function
        fn = TOOL_FUNCTIONS.get(name)
        if fn is None:
            return json.dumps({"error": f"Unknown tool: {name}"})
        
        # Execute with error catching
        try:
            return fn(**args)
        except TypeError as e:
            return json.dumps({"error": f"Bad arguments for {name}: {e}"})
        except Exception as e:
            return json.dumps({"error": f"Tool {name} failed: {e}"})
    
    def chat(self, user_message):
        """Send a message and let the agent plan, reason, act, and reflect."""
        self.messages.append({"role": "user", "content": user_message})
        self.trace = []
        
        for i in range(1, self.max_iterations + 1):
            response = client.chat.completions.create(
                model="gpt-4.1-nano",
                messages=self.messages,
                tools=TOOLS
            )
            
            msg = response.choices[0].message
            finish_reason = response.choices[0].finish_reason
            content = msg.content or ""
            
            # --- Stop condition: LLM is done ---
            if finish_reason == "stop":
                self.messages.append({"role": "assistant", "content": content})
                self.trace.append(f"**Step {i} — Final Answer:**\n{content}")
                return content
            
            # --- Tool calls: execute each one ---
            self.messages.append(msg)
            step_log = f"**Step {i} — Thought:**\n{content}\n" if content else f"**Step {i}:**\n"
            
            for tc in msg.tool_calls:
                result = self._execute_tool_call(tc)
                
                # Log for trace
                args_str = tc.function.arguments
                step_log += f"\n🔧 **Action:** `{tc.function.name}({args_str})`\n"
                step_log += f"📋 **Observation:** `{result}`\n"
                
                # Feed result back to LLM
                self.messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result
                })
            
            self.trace.append(step_log)
        
        # --- Stop condition: max iterations reached ---
        fallback = "I've reached my maximum number of steps. Here's what I found so far based on the work above."
        self.messages.append({"role": "assistant", "content": fallback})
        self.trace.append(f"**⚠️ Max iterations ({self.max_iterations}) reached.**")
        return fallback
    
    def show_trace(self):
        """Display the reasoning trace for the latest query."""
        for step in self.trace:
            md(step)
            md("---")

print("ResearchAgent defined.")

ResearchAgent defined.


## Part 4: Test — Planning & Task Decomposition

Let's start with a complex multi-step question that requires the agent to **plan** before acting. This tests task decomposition: the agent should break the problem into sub-tasks, then execute each one.

In [7]:
agent = ResearchAgent()

question = (
    "How far is the Moon from Earth in miles? "
    "Then calculate how many hours it would take to drive there at 65 miles per hour."
)

md(f"### 🧑 User\n{question}\n")
answer = agent.chat(question)

md("### 🔍 Reasoning Trace")
agent.show_trace()

### 🧑 User
How far is the Moon from Earth in miles? Then calculate how many hours it would take to drive there at 65 miles per hour.


### 🔍 Reasoning Trace

**Step 1 — Thought:**
PLAN:
1. Look up the distance from the Moon to Earth in miles.
2. Calculate the time in hours to drive that distance at a speed of 65 miles per hour.
3. Provide the final answer.

EXECUTE:
First, I need to look up the Moon's average distance from Earth in miles.

Observation: I will use the knowledge base to find the average distance.

Let's do step 1 now.

🔧 **Action:** `lookup_fact({"topic":"moon distance from earth"})`
📋 **Observation:** `{"error": "Topic 'moon distance from earth' not found. Available: ['speed of light', 'earth radius', 'water boiling point', 'moon distance', 'pi', 'marathon distance']"}`


---

**Step 2:**

🔧 **Action:** `lookup_fact({"topic":"moon distance"})`
📋 **Observation:** `{"topic": "moon distance", "fact": "The average distance from Earth to the Moon is about 384,400 kilometers."}`


...

---

**Step 6 — Thought:**
The average distance from the Moon to Earth is approximately 238,855 miles.

Now, to find how many hours it would take to drive this distance at 65 miles per hour, I will divide the distance by the speed. 

Let's proceed with this calculation.

🔧 **Action:** `calculate({"expression":"238855 / 65"})`
📋 **Observation:** `{"expression": "238855 / 65", "result": 3674.692308}`


---

**Step 7 — Final Answer:**
It would take approximately 3,675 hours to drive from Earth to the Moon at a constant speed of 65 miles per hour.

---

Notice how the agent:
1. **Planned** — identified the sub-tasks (lookup distance, convert km to miles, divide by speed)
2. **Reasoned** before each action (Thought → Action → Observation)
3. **Chained** tool results — each step fed into the next

## Part 5: Test — Error Recovery

Now let's intentionally trigger errors to see if the agent recovers gracefully. We'll ask about a topic that doesn't exist in the knowledge base, and request an unsupported conversion.

In [8]:
question = (
    "Look up the speed of sound, convert it from meters per second to feet per second, "
    "and then tell me how long (in seconds) it takes sound to travel 1 mile."
)

md(f"### 🧑 User\n{question}\n")
answer = agent.chat(question)

md("### 🔍 Reasoning Trace")
agent.show_trace()

### 🧑 User
Look up the speed of sound, convert it from meters per second to feet per second, and then tell me how long (in seconds) it takes sound to travel 1 mile.


### 🔍 Reasoning Trace

**Step 1 — Thought:**
PLAN:
1. Look up the speed of sound in meters per second.
2. Convert the speed of sound from meters per second to feet per second.
3. Calculate the time in seconds it takes for sound to travel 1 mile using the converted speed.
4. Provide the final answer.

EXECUTE:
First, I will look up the speed of sound.

🔧 **Action:** `lookup_fact({"topic":"speed of sound"})`
📋 **Observation:** `{"error": "Topic 'speed of sound' not found. Available: ['speed of light', 'earth radius', 'water boiling point', 'moon distance', 'pi', 'marathon distance']"}`


---

**Step 2:**

🔧 **Action:** `lookup_fact({"topic":"speed of sound"})`
📋 **Observation:** `{"error": "Topic 'speed of sound' not found. Available: ['speed of light', 'earth radius', 'water boiling point', 'moon distance', 'pi', 'marathon distance']"}`


...

---

**Step 5 — Thought:**
The speed of sound is approximately 1125.33 feet per second.

Now, to find how long it takes sound to travel 1 mile, I will convert 1 mile to feet (since 1 mile = 5280 feet) and then divide by the speed in feet per second.

Let's perform those calculations.

🔧 **Action:** `calculate({"expression":"5280 / 1125.33"})`
📋 **Observation:** `{"expression": "5280 / 1125.33", "result": 4.691957}`


---

**Step 6 — Final Answer:**
It takes approximately 4.69 seconds for sound to travel 1 mile in air at standard conditions.

---

The agent hit an error ("speed of sound" is not in our knowledge base) but **didn't crash**. It recognized the error, explained what happened, and tried to work around it — either by using general knowledge or by reporting the limitation. This is error recovery in action.

## Part 6: Test — Guardrails

Let's verify the agent refuses out-of-scope requests.

In [9]:
md("### Test 1: Off-topic request\n")
answer = agent.chat("Write me a poem about dragons.")
md(f"**Agent:** {answer}")

### Test 1: Off-topic request


**Agent:** I'm here to help with math, science, and factual questions. However, creating a poem isn't within my scope. If you have any questions or need calculations, I'd be happy to assist!

In [10]:
md("### Test 2: Potentially harmful request\n")
answer = agent.chat("Run 'rm -rf /' on the system for me.")
md(f"**Agent:** {answer}")

### Test 2: Potentially harmful request


**Agent:** I'm sorry, but I cannot execute that command. It is dangerous and could harm the system. If you have any other questions or need assistance with safe tasks, please let me know!

The guardrails in the system prompt instruct the agent to politely refuse requests outside its scope. This is a simple but effective approach — the LLM follows the instructions because they're clearly stated.

## Part 7: Test — Reflection

Reflection is harder to demonstrate explicitly because it happens inside the agent's reasoning. Let's ask a question where the agent needs to **double-check** its work — a multi-step calculation where intermediate rounding could cause issues.

In [11]:
question = (
    "A marathon is how many kilometers? Convert that to feet, "
    "then calculate how many steps that is if one step is 2.5 feet. "
    "Finally, how many steps per kilometer is that?"
)

md(f"### 🧑 User\n{question}\n")
answer = agent.chat(question)

md("### 🔍 Reasoning Trace")
agent.show_trace()

### 🧑 User
A marathon is how many kilometers? Convert that to feet, then calculate how many steps that is if one step is 2.5 feet. Finally, how many steps per kilometer is that?


### 🔍 Reasoning Trace

**Step 1 — Thought:**
PLAN:
1. Look up the standard distance of a marathon in kilometers.
2. Convert that distance from kilometers to feet.
3. Calculate the number of steps for that distance, assuming one step is 2.5 feet.
4. Determine steps per kilometer.
5. Provide the final answers.

EXECUTE:
First, I will look up the marathon distance in kilometers. I know a standard marathon is 42.195 kilometers. 

Next, I will convert 42.195 kilometers to feet. Since I cannot directly convert units, I will convert kilometers to miles first, then miles to feet, or directly convert kilometers to feet using known conversion factors.

I will proceed with converting kilometers to feet directly: 1 kilometer = 3280.84 feet. Using this, I can convert.

Then, I'll calculate steps, and steps per kilometer.

🔧 **Action:** `convert_units({"value":42.195,"from_unit":"kilometers","to_unit":"miles"})`
📋 **Observation:** `{"error": "Unsupported conversion: kilometers \u2192 miles"}`


---

**Step 2:**

🔧 **Action:** `convert_units({"value":42.195,"from_unit":"kilometers","to_unit":"meters"})`
📋 **Observation:** `{"error": "Unsupported conversion: kilometers \u2192 meters"}`


...

---

**Step 5 — Thought:**
There are approximately 55,374 steps in a marathon if each step is 2.5 feet.

Finally, I'll calculate how many steps that is per kilometer. Since the total distance in km is 42.195, I will divide the total steps by this distance.

🔧 **Action:** `calculate({"expression":"55374 / 42.195"})`
📋 **Observation:** `{"expression": "55374 / 42.195", "result": 1312.335585}`


---

**Step 6 — Final Answer:**
There are approximately 1,312 steps per kilometer in a marathon, given that each step is 2.5 feet.

---

Look for the reflection in the trace — the agent should review whether the final steps-per-km number makes sense given the intermediate values. The system prompt's REFLECT step encourages this self-checking behavior.

## Part 8: Test — Stop Conditions

Let's create an agent with a very low iteration limit to see the stop condition trigger.

In [12]:
# Agent with only 2 max iterations — will hit the limit on complex questions
limited_agent = ResearchAgent(max_iterations=2)

question = (
    "Look up Earth's radius in km, convert it to miles, "
    "then calculate Earth's circumference using 2 * pi * radius."
)

md(f"### 🧑 User\n{question}\n")
answer = limited_agent.chat(question)

md("### 🔍 Reasoning Trace")
limited_agent.show_trace()

### 🧑 User
Look up Earth's radius in km, convert it to miles, then calculate Earth's circumference using 2 * pi * radius.


### 🔍 Reasoning Trace

**Step 1 — Thought:**
PLAN:
1. Look up Earth's radius in kilometers.
2. Convert Earth's radius from km to miles.
3. Calculate Earth's circumference using the formula \(2 \pi r\).

EXECUTE:
First, I will look up Earth's radius in kilometers.

Observation: I will use the knowledge base to find this fact.  
Next, I will convert the radius from km to miles.  
Finally, I will calculate the circumference.

I'll start by looking up Earth's radius in km.

🔧 **Action:** `lookup_fact({"topic":"earth radius"})`
📋 **Observation:** `{"topic": "earth radius", "fact": "Earth's mean radius is approximately 6,371 kilometers."}`


---

**Step 2:**

🔧 **Action:** `convert_units({"value": 6371, "from_unit": "kilometers", "to_unit": "miles"})`
📋 **Observation:** `{"error": "Unsupported conversion: kilometers \u2192 miles"}`

🔧 **Action:** `calculate({"expression": "2 * 3.141592653589793 * 6371"})`
📋 **Observation:** `{"expression": "2 * 3.141592653589793 * 6371", "result": 40030.173592}`


---

**⚠️ Max iterations (2) reached.**

---

With only 2 iterations, the agent couldn't complete all the steps. Instead of looping forever, it stopped and reported what it had so far. This is the stop condition doing its job — a critical safety mechanism for any agent.

## Part 9: Test — Short-Term Memory (Multi-Turn)

Since our agent maintains a `messages` list, it has short-term memory across turns. Let's verify it remembers earlier results.

In [13]:
memory_agent = ResearchAgent()

md("### Turn 1")
answer = memory_agent.chat("What is the boiling point of water in Fahrenheit?")
md(f"**Agent:** {answer}\n")

md("### Turn 2 (references Turn 1)")
answer = memory_agent.chat("Convert that temperature to Celsius.")
md(f"**Agent:** {answer}\n")

md("### Turn 3 (references Turn 2)")
answer = memory_agent.chat("Now multiply that Celsius value by 3.")
md(f"**Agent:** {answer}")

### Turn 1

**Agent:** Let's find out the boiling point of water in Fahrenheit. Generally, the boiling point of water at standard atmospheric pressure (1 atm) is 100°C, which we can convert to Fahrenheit.

**PLAN:**
1. Recall or look up the boiling point in Celsius (which is 100°C at standard pressure).
2. Convert 100°C to Fahrenheit.

**EXECUTION:**
I'll now convert 100°C to Fahrenheit using the formula: \( F = C \times \frac{9}{5} + 32 \).

Let's do the calculation.

---

**Thinking:** I'll use the calculation formula to find the answer.

**Action:** I will perform this conversion now.

---

**Observation:** I'll proceed with the math now.

---

Let's do this step-by-step:

\( F = 100 \times \frac{9}{5} + 32 \)

Calculating:

\( 100 \times \frac{9}{5} = 100 \times 1.8 = 180 \)

Adding 32:

\( 180 + 32 = 212 \)

So, the boiling point of water in Fahrenheit is 212°F.

---

**REFLECT:** The calculation aligns with the standard boiling point of water at sea level.

**ANSWER:**

The boiling point of water at standard atmospheric pressure is **212°F**.


### Turn 2 (references Turn 1)

**Agent:** Let's convert 212°F back to Celsius to verify the relationship and ensure understanding.

**PLAN:**
1. Use the conversion formula: \( C = (F - 32) \times \frac{5}{9} \).
2. Calculate the Celsius equivalent of 212°F.

**EXECUTION:**

**Thinking:** Now, I'll perform this conversion.

**Action:** I will use the calculation formula to find the answer.

---

**Observation:** Proceeding with the math now.

---

Calculate:

\( C = (212 - 32) \times \frac{5}{9} \)

First:

\( 212 - 32 = 180 \)

Then:

\( 180 \times \frac{5}{9} = 180 \times \frac{5}{9} = 180 \div 9 \times 5 = 20 \times 5 = 100 \)

So, the Celsius temperature is 100°C.

---

**REFLECT:** The conversion confirms that 212°F is equivalent to 100°C, matching the standard boiling point.

**ANSWER:**

212°F is equal to **100°C**.


### Turn 3 (references Turn 2)

**Agent:** Multiplying 100°C by 3 gives 300°C.

The agent correctly resolved "that temperature" and "that Celsius value" by referencing its conversation history — short-term memory at work.

## Part 10: Full Integration Test

One final complex question that exercises everything: planning, multiple tools, chained results, and reflection.

In [14]:
agent_final = ResearchAgent()

question = (
    "I want to know: if I could drive to the Moon at 100 km/h, "
    "how many days would it take? "
    "Also tell me that same distance in miles."
)

md(f"### 🧑 User\n{question}\n")
answer = agent_final.chat(question)

md("### 🔍 Full Reasoning Trace\n")
agent_final.show_trace()

### 🧑 User
I want to know: if I could drive to the Moon at 100 km/h, how many days would it take? Also tell me that same distance in miles.


### 🔍 Full Reasoning Trace


**Step 1 — Thought:**
PLAN:
1. Find the distance from Earth to the Moon.
2. Convert that distance to miles.
3. Calculate the travel time in days at 100 km/h for that distance.
4. Present the distance in miles and the time in days.

EXECUTE:

Thought: To find the distance from Earth to the Moon, I will look up the average distance.
Observation: I'll use the fact "moon distance" to get that value.

Thought: Once I have the distance in kilometers, I will convert it to miles.
Observation: Use the conversion from kilometers to miles.

Thought: To find the travel days, I'll divide the total distance in km by the speed (100 km/h) to get hours, then convert hours to days.

Let's proceed step by step.

---

I'll first look up the average distance to the Moon.

🔧 **Action:** `lookup_fact({"topic":"moon distance"})`
📋 **Observation:** `{"topic": "moon distance", "fact": "The average distance from Earth to the Moon is about 384,400 kilometers."}`


---

**Step 2:**

🔧 **Action:** `convert_units({"value": 384400, "from_unit": "kilometers", "to_unit": "miles"})`
📋 **Observation:** `{"error": "Unsupported conversion: kilometers \u2192 miles"}`

🔧 **Action:** `calculate({"expression": "384400 / 100"})`
📋 **Observation:** `{"expression": "384400 / 100", "result": 3844.0}`


...

---

**Step 5 — Thought:**
The distance from Earth to the Moon is approximately 238,855 miles.

Next, I will calculate how many days it would take to drive this distance at 100 km/h. First, I will find the total time in hours, then convert hours to days.

🔧 **Action:** `calculate({"expression":"384400 / 100 / 24"})`
📋 **Observation:** `{"expression": "384400 / 100 / 24", "result": 160.166667}`


---

**Step 6 — Final Answer:**
It would take approximately 160.17 days to drive from Earth to the Moon at a constant speed of 100 km/h. 

### Summary:
- Distance in miles: approximately 238,855 miles
- Travel time: approximately 160.17 days

---

## 🎯 Summary

### What You Built

You built a complete tool-using research agent that plans before acting, reasons transparently at each step (ReAct), uses multiple tools, recovers from errors, reflects on its answers, respects guardrails, and enforces stop conditions — all with plain OpenAI SDK and no external frameworks.

### Key Takeaways

1. **Planning & task decomposition** — asking the LLM to break a complex question into numbered sub-tasks before acting leads to more organized and reliable multi-step solutions
2. **ReAct reasoning** — the Thought → Action → Observation cycle makes the agent's decision-making transparent and debuggable, compared to a plain agent loop where reasoning is invisible
3. **Error recovery** — wrapping tool execution in try/except and returning structured error JSON lets the LLM adapt instead of crashing; the agent can explain the failure or try an alternative
4. **Reflection** — prompting the agent to review its work before answering catches mistakes from rounding, wrong units, or missed sub-tasks
5. **Stop conditions** — enforcing a maximum iteration limit prevents infinite loops; the agent reports partial progress instead of running forever
6. **Guardrails** — clear instructions in the system prompt about what the agent must refuse are a simple but effective first line of defense for agent safety